# Ingredient processing: every distinct ingredient line -> English name, amount, unit

1. **Collect** every distinct ingredient line (lowercase, whitespace collapsed) into `ingredient_lines`, with how many
   times it occurs. Re-running only adds new lines and updates the counts.
2. **Process** them with Gemini, most common lines first, on every key in parallel. Each answer is saved right away,
   so the loop can be interrupted (stop button) and restarted at any time. It respects the daily quota per key
   and stops by itself once every key has used up today's quota: run it again after the reset.
3. **Map back**: `recipe_ingredients` gets one row per ingredient of every recipe (recipe, position, name, amount, unit),
   with units cleaned up locally. Can be rebuilt any time, also while step 2 is still running.

Keep the Mac awake while it runs, e.g. `caffeinate -i` in a terminal.

In [9]:
import json
import sqlite3
import threading
import time
from collections import Counter, deque
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from google import genai
from google.genai import errors, types

DB_PATH = "data.nosync/recipes.db"
USAGE_LOG = "data.nosync/gemini_usage.jsonl"  # shared with gemini_quality_test.ipynb, used to count today's requests

MODELS = ["gemini-3.5-flash-lite"]  # one lane per key x model; add "gemini-3.1-flash-lite" for more quota
THINKING_LEVEL = "minimal"
BATCH_SIZE = 650  # lines per request: ~18.5K output tokens, well below MAX_OUTPUT_TOKENS
MAX_OUTPUT_TOKENS = 32768
MAX_LINE_CHARS = 200  # longer lines are almost always prose, not ingredients: skipped
MAX_ATTEMPTS = 2  # a line the model leaves out this often is marked failed (status -1)

# Free tier limits per key (= per Google Cloud project) and model. Check them in AI Studio.
RPD = 500  # 500 on the free tier; None = no own limit, keep going until Google answers with 429
RPM = 14 # our own pace per lane (the free tier allows 15)
IN_FLIGHT = 200  # requests at the same time per lane, so the ~1 min per request does not slow us below RPM
QUOTA_TZ = ZoneInfo("America/Los_Angeles")  # daily quotas reset at midnight Pacific time

# One key per line in gemini_keys.txt, ONE KEY PER PROJECT (keys of the same project share a quota).
with open("gemini_keys.txt") as f:
    API_KEYS = [line.strip() for line in f if line.strip() and not line.startswith("#")]
API_KEYS = list(dict.fromkeys(API_KEYS))
clients = [genai.Client(api_key=key) for key in API_KEYS]
print(f"{len(clients)} keys x {len(MODELS)} models")

con = sqlite3.connect(DB_PATH, timeout=60)
# status: 0 = to do, 1 = done, -1 = model left it out MAX_ATTEMPTS times, -2 = skipped (too long)
# items: JSON [[name, amount, unit], ...] exactly as the model returned them; cleanup happens in step 3
con.executescript("""
CREATE TABLE IF NOT EXISTS ingredient_lines (
    id INTEGER PRIMARY KEY,
    line TEXT NOT NULL UNIQUE,
    n INTEGER NOT NULL,
    status INTEGER NOT NULL DEFAULT 0,
    items TEXT,
    model TEXT,
    done_at REAL
);
CREATE INDEX IF NOT EXISTS ingredient_lines_todo ON ingredient_lines(status, n DESC);
""")

5 keys x 1 models


In [10]:
UNITS = ["g", "kg", "mg", "ml", "cl", "dl", "l", "tsp", "tbsp", "cup", "oz", "lb", "fl oz", "pinch", "dash", "piece",
         "clove", "slice", "sheet", "can", "jar", "package", "bunch", "sprig", "stalk", "leaf", "handful", "drop"]

PROMPT = f"""You turn recipe ingredient lines into structured data.

The input is a JSON array of [index, line]. The lines can be in any language.
For every line return one or more objects {{"i": index, "en": name, "a": amount, "u": unit}}:
- en: the ingredient name TRANSLATED INTO ENGLISH, lowercase, singular. Always English, also when the line is a
  single word: never copy a word from the input unless it is also the English word ("avocado", "curry").
  Only the ingredient: no quantity, unit, brand, size or notes.
  Drop preparation and state words (fresh, frozen, dried, ground, chopped, minced, sliced, grated, peeled, ...),
  keep words that make it a different ingredient ("brown sugar", "egg white", "soy sauce", "coconut milk").
- a: the amount as a number. Fractions as decimals ("1½" -> 1.5), ranges as the first number ("2-3" -> 2), null if none.
- u: the unit in English, singular. Use one of {", ".join(UNITS)} if it fits,
  otherwise a short English word. null if there is none; plain counts ("3 eggs") have no unit.
- Never use "spoon". A big, soup or table spoon is "tbsp" ("el", "ek", "msk", "spl", "ss", "c.s.", "ст. л.", "łyżka",
  "lingura", "colher de sopa", "sudu besar", "כף", "大さじ"). A small or tea spoon is "tsp" ("tl", "tsk", "c.c.",
  "ч. л.", "łyżeczka", "colher de chá", "sudu kecil", "小さじ").
- "gr" -> g. Counting words ("stk", "ks", "db", "szt", "шт", "adet", "biji", "個") -> piece. Tins ("lata", "dose") -> can.
- Alternatives ("butter or margarine"): only the first one.
- Several ingredients in one line ("salt and pepper"): one object each, with the same i.
- Not an ingredient (headings like "for the sauce:", equipment, tips): {{"i": index, "en": null, "a": null, "u": null}}.
Every index must appear in the output.

Example input:  [[0, "2 cups chopped onions"], [1, "salz und pfeffer"], [2, "für die soße:"], [3, "1 el sojasauce"],
                 [4, "3 eier"], [5, "1 cebolla"], [6, "czosnek"], [7, "1 kg брашно"], [8, "水 2大匙"], [9, "1 ks vejce"],
                 [10, "2 łyżki oleju"]]
Example output: [{{"i":0,"en":"onion","a":2,"u":"cup"}},{{"i":1,"en":"salt","a":null,"u":null}},{{"i":1,"en":"pepper","a":null,"u":null}},{{"i":2,"en":null,"a":null,"u":null}},{{"i":3,"en":"soy sauce","a":1,"u":"tbsp"}},{{"i":4,"en":"egg","a":3,"u":null}},{{"i":5,"en":"onion","a":1,"u":"piece"}},{{"i":6,"en":"garlic","a":null,"u":null}},{{"i":7,"en":"flour","a":1,"u":"kg"}},{{"i":8,"en":"water","a":2,"u":"tbsp"}},{{"i":9,"en":"egg","a":1,"u":"piece"}},{{"i":10,"en":"oil","a":2,"u":"tbsp"}}]

Input:
"""

RESPONSE_SCHEMA = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "i": {"type": "integer"},
            "en": {"type": ["string", "null"]},
            "a": {"type": ["number", "null"]},
            "u": {"type": ["string", "null"]},
        },
        "required": ["i", "en", "a", "u"],
    },
}

In [11]:
# Step 1: collect the distinct lines. ~1 min. Safe to re-run: adds new lines, updates counts, keeps results.
def norm(line):
    """The key lines are deduplicated on: lowercase, whitespace collapsed."""
    return " ".join(str(line).split()).lower()


t = time.time()
counts = Counter()
for (ingredients,) in con.execute("SELECT ingredients FROM recipes"):
    for line in json.loads(ingredients):
        if line := norm(line):
            counts[line] += 1
with con:
    con.executemany(
        "INSERT INTO ingredient_lines (line, n) VALUES (?, ?) ON CONFLICT(line) DO UPDATE SET n = excluded.n",
        counts.items(),
    )
    con.execute("UPDATE ingredient_lines SET status = -2 WHERE status = 0 AND length(line) > ?", (MAX_LINE_CHARS,))
del counts
print(f"{time.time() - t:.0f} s")
print(con.execute("SELECT status, COUNT(*), SUM(n) FROM ingredient_lines GROUP BY 1").fetchall())

29 s
[(-2, 1374, 1391), (-1, 1, 1), (0, 94432, 94432), (1, 1969444, 5944650)]


In [12]:
# --- quota bookkeeping (the API does not report how much is left, so every request is logged)
def quota_day(ts=None):
    return datetime.fromtimestamp(ts or time.time(), QUOTA_TZ).date().isoformat()


def next_reset():
    """Next quota reset (midnight Pacific) in local time."""
    now = datetime.now(QUOTA_TZ)
    return datetime.combine(now.date() + timedelta(days=1), datetime.min.time(), QUOTA_TZ).astimezone()


def seconds_to_reset():
    now = datetime.now(QUOTA_TZ)
    midnight = datetime.combine(now.date() + timedelta(days=1), datetime.min.time(), QUOTA_TZ)
    return (midnight - now).total_seconds() + 120  # 2 min margin


_log_lock = threading.Lock()


def log_usage(rec):
    with _log_lock, open(USAGE_LOG, "a") as f:
        f.write(json.dumps(rec) + "\n")


def transient(rec):
    """Failed for reasons that have nothing to do with the lines: overloaded, quota, network."""
    error = rec.get("error", "")
    return error.startswith(("5", "429", "network"))


def used_today():
    used = Counter()
    try:
        with open(USAGE_LOG) as f:
            for line in f:
                rec = json.loads(line)
                # Every attempt counts, also 503s: AI Studio showed them in the 500/day (2026-10-29).
                if rec["day"] == quota_day():
                    used[rec["key"], rec["model"]] += 1
    except FileNotFoundError:
        pass
    return used


# --- one request, with retries for overloaded servers and network errors
RETRY_WAIT = 15  # 15, 30, 60, 120, 240 s
MAX_RETRIES = 5


def call(key_no, model, lines):
    """Returns (parsed JSON or None, record). Runs in a worker thread, never touches the database."""
    payload = json.dumps([[i, line] for i, line in enumerate(lines)], ensure_ascii=False)
    config = types.GenerateContentConfig(
        temperature=0,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        response_mime_type="application/json",
        response_json_schema=RESPONSE_SCHEMA,
        thinking_config=types.ThinkingConfig(thinking_level=THINKING_LEVEL) if THINKING_LEVEL else None,
    )
    rec = {"day": quota_day(), "key": key_no, "model": model, "thinking": THINKING_LEVEL, "lines": len(lines)}
    for attempt in range(MAX_RETRIES + 1):
        t = time.time()
        try:
            response = clients[key_no].models.generate_content(model=model, contents=PROMPT + payload, config=config)
            break
        except errors.APIError as e:
            rec.update(ts=time.time(), latency=time.time() - t, retries=attempt, error=f"{e.code} {e.message}"[:300])
            if e.code == 429:  # says which limit was hit (per minute / per day, requests / tokens)
                rec["details"] = json.dumps(e.details, ensure_ascii=False)[:2000]
            retry = e.code is not None and e.code >= 500
        except Exception as e:  # connection reset, timeout, ...
            rec.update(ts=time.time(), latency=time.time() - t, retries=attempt, error=f"network {e!r}"[:300])
            retry = True
        log_usage(rec)
        if not retry or attempt == MAX_RETRIES:
            return None, rec
        time.sleep(RETRY_WAIT * 2 ** attempt)
    rec.pop("error", None)
    usage = response.usage_metadata
    finish = response.candidates[0].finish_reason if response.candidates else None
    rec.update(
        ts=time.time(),
        latency=time.time() - t,
        retries=attempt,
        prompt_tokens=usage.prompt_token_count or 0,
        thoughts_tokens=usage.thoughts_token_count or 0,
        output_tokens=usage.candidates_token_count or 0,
        cached_tokens=usage.cached_content_token_count or 0,
        total_tokens=usage.total_token_count or 0,
        finish=getattr(finish, "name", finish),
    )
    try:
        result = json.loads(response.text)
    except (json.JSONDecodeError, TypeError) as e:  # e.g. cut off at MAX_TOKENS
        rec["error"] = f"bad json: {e}"[:300]
        result = None
    log_usage(rec)
    return result, rec


def parse(n_lines, result):
    """{line index: [[name, amount, unit], ...]} for every line the model answered."""
    out = {}
    for item in result if isinstance(result, list) else []:
        i = item.get("i") if isinstance(item, dict) else None
        if isinstance(i, int) and 0 <= i < n_lines:
            out.setdefault(i, []).append([item.get("en"), item.get("a"), item.get("u")])
    return out


class Lane:
    """One key x model with its own daily quota, rate limit and pauses."""

    def __init__(self, key_no, model, used):
        self.key_no, self.model = key_no, model
        self.name = f"key{key_no + 1}/{model}"
        self.day, self.used = quota_day(), used
        self.in_flight = 0
        self.next_at = 0.0  # rate limit
        self.paused_until = 0.0  # after a 429 or when the daily quota is used up
        self.n_429 = 0
        self.exhausted = False  # today's quota is used up

    def ready(self, now):
        if quota_day() != self.day:  # new quota day
            self.day, self.used, self.exhausted = quota_day(), 0, False
        if RPD and self.used >= RPD and not self.exhausted:
            self.exhausted = True
            self.paused_until = now + seconds_to_reset()
            print(f"{self.name}: {RPD} requests used today, pausing {self.paused_until - now:,.0f} s until the reset")
        return self.in_flight < IN_FLIGHT and now >= max(self.next_at, self.paused_until)

    def got_429(self, now, details=""):
        if now < self.paused_until:  # several requests were in flight: count one 429 per pause, not each of them
            return
        self.n_429 += 1
        if "PerDay" in details:  # the daily quota of this project is used up
            self.exhausted = True
            self.paused_until = now + seconds_to_reset()
        else:  # short-term throttling (no quotaId, or per minute): 2, 4, 8, 16, 30, 30 ... min
            self.paused_until = now + min(60 * 2 ** self.n_429, 1800)
        print(f"{self.name}: 429 quota exceeded (#{self.n_429}), pausing {(self.paused_until - now) / 60:,.0f} min")


def save(answered, failed, model):
    with con:
        con.executemany(
            "UPDATE ingredient_lines SET status = 1, items = ?, model = ?, done_at = ? WHERE line = ?",
            [(json.dumps(items, ensure_ascii=False), model, time.time(), line) for line, items in answered],
        )
        con.executemany("UPDATE ingredient_lines SET status = -1 WHERE line = ?", [(line,) for line in failed])


print("used today:", {f"key{k + 1}/{m}": n for (k, m), n in used_today().items()} or "nothing")

used today: {'key3/gemini-3.5-flash-lite': 500, 'key1/gemini-3.5-flash-lite': 500, 'key2/gemini-3.5-flash-lite': 493}


In [13]:
# Optional: one small batch, to see that everything works before starting the long run. Saves its results.
if False:
    lines = [line for (line,) in con.execute(
        "SELECT line FROM ingredient_lines WHERE status = 0 ORDER BY n DESC LIMIT 20")]
    result, rec = call(0, MODELS[0], lines)
    print(rec)
    parsed = parse(len(lines), result)
    for i, line in enumerate(lines):
        print(f"{line[:60]:60} -> {parsed.get(i)}")
    save([(line, parsed[i]) for i, line in enumerate(lines) if i in parsed], [], MODELS[0])

In [ ]:
# Step 2: the long run. Stops by itself once every key has used up today's quota: run it again after the reset.
# Stop button = finish the requests in flight, save them, then stop. Re-run to continue.
todo = deque((line, 0) for (line,) in con.execute(
    "SELECT line FROM ingredient_lines WHERE status = 0 ORDER BY n DESC"))  # (line, failed attempts so far)
retry = deque()  # lines to send again: the model left them out, or the request failed
total = len(todo)
used = used_today()
lanes = [Lane(k, m, used[k, m]) for m in MODELS for k in range(len(clients))]
STAGGER = 5  # seconds between the first request of each lane, so not every key starts at the same moment
for i, lane in enumerate(lanes):
    lane.next_at = time.monotonic() + i * STAGGER
print(f"{total:,} lines to do, {len(lanes)} lanes, used today: {[l.used for l in lanes]}")


def take_batch():
    batch = []
    while len(batch) < BATCH_SIZE and (retry or todo):
        batch.append((retry or todo).popleft())
    return batch


started, done_lines, failed_lines, stopping = time.time(), 0, 0, False
tokens = Counter()
futures = {}
with ThreadPoolExecutor(len(lanes) * IN_FLIGHT) as pool:
    while futures or (not stopping and (todo or retry)):
        try:
            now = time.monotonic()
            for lane in lanes:
                while not stopping and (todo or retry) and lane.ready(now):
                    batch = take_batch()
                    lane.in_flight += 1
                    lane.used += 1
                    lane.next_at = now + 1.05 * 60 / RPM
                    futures[pool.submit(call, lane.key_no, lane.model, [l for l, _ in batch])] = (lane, batch)
            if not futures and all(l.exhausted for l in lanes):
                print(f"{datetime.now():%a %H:%M} every key has used up today's quota. "
                      f"Run this cell again after the reset: {next_reset():%a %H:%M}")
                break
            # Until the next lane may send again (rate limit or pause), but at most 30 s.
            wake = min((max(l.next_at, l.paused_until) for l in lanes if not l.exhausted), default=time.monotonic() + 30) - time.monotonic()
            if not futures:  # nothing in flight: every lane is waiting
                if wake > 60:
                    print(f"{datetime.now():%a %H:%M} all lanes paused, sleeping {wake / 60:,.0f} min")
                time.sleep(min(max(wake, 0.1), 600))
                continue
            finished, _ = wait(futures, timeout=min(max(wake, 0.1), 30), return_when=FIRST_COMPLETED)
        except KeyboardInterrupt:
            if stopping:
                raise
            stopping = True
            print(f"stopping: waiting for {len(futures)} requests in flight (stop again to abandon them)")
            continue

        for future in finished:
            lane, batch = futures.pop(future)
            lane.in_flight -= 1
            result, rec = future.result()
            error = rec.get("error", "")
            lane.used += rec.get("retries", 0)  # every retry inside call() was a request too
            if error.startswith("429"):
                lane.got_429(time.monotonic(), rec.get("details", ""))
            elif "error" not in rec:
                lane.n_429 = 0
            if result is None and transient(rec):
                retry.extendleft(reversed(batch))  # not the lines' fault: same attempt count
                print(f"{lane.name}: {error[:100]}, batch goes back in the queue")
                if "details" in rec:
                    print("   ", ", ".join(sorted(set(__import__("re").findall(r'"quotaId": "([^"]+)"', rec["details"])))))
                continue
            parsed = parse(len(batch), result)
            answered, failed = [], []
            for i, (line, attempts) in enumerate(batch):
                if i in parsed:
                    answered.append((line, parsed[i]))
                elif attempts + 1 >= MAX_ATTEMPTS:
                    failed.append(line)
                else:
                    retry.append((line, attempts + 1))
            save(answered, failed, lane.model)
            done_lines += len(answered)
            failed_lines += len(failed)
            for k in ("prompt_tokens", "thoughts_tokens", "output_tokens"):
                tokens[k] += rec.get(k, 0)
            rate = done_lines / (time.time() - started)
            left = len(todo) + len(retry)
            print(
                f"{datetime.now():%a %H:%M} {lane.name}: {len(answered)}/{len(batch)} in {rec['latency']:.0f} s "
                f"{error[:60]}| done {done_lines:,}/{total:,} ({done_lines / max(total, 1):.1%}), failed {failed_lines}, "
                f"left {left:,} | today {[l.used for l in lanes]} | {rate * 3600:,.0f} lines/h"
            )

print(f"\nfinished: {done_lines:,} done, {failed_lines} failed, tokens {dict(tokens)}")
print(con.execute("SELECT status, COUNT(*), SUM(n) FROM ingredient_lines GROUP BY 1").fetchall())

94,432 lines to do, 5 lanes, used today: [500, 493, 500, 0, 0]
key1/gemini-3.5-flash-lite: 500 requests used today, pausing 74,877 s until the reset
key3/gemini-3.5-flash-lite: 500 requests used today, pausing 74,877 s until the reset
key2/gemini-3.5-flash-lite: 429 quota exceeded (#1), pausing 1,248 min
key2/gemini-3.5-flash-lite: 429 You exceeded your current quota, please check your plan and billing details. For more informatio, batch goes back in the queue
    GenerateRequestsPerDayPerProjectPerModel-FreeTier
Fri 11:14 key4/gemini-3.5-flash-lite: 650/650 in 26 s | done 650/94,432 (0.7%), failed 0, left 84,682 | today [500, 494, 500, 8, 7] | 46,634 lines/h
Fri 11:14 key4/gemini-3.5-flash-lite: 650/650 in 35 s | done 1,300/94,432 (1.4%), failed 0, left 84,682 | today [500, 494, 500, 8, 7] | 93,190 lines/h
Fri 11:15 key4/gemini-3.5-flash-lite: 650/650 in 39 s | done 1,950/94,432 (2.1%), failed 0, left 82,082 | today [500, 494, 500, 10, 9] | 120,828 lines/h
Fri 11:15 key5/gemini-3.5-fl

In [ ]:
# Progress, any time (also from a second notebook while step 2 runs).
rows = dict(((s, (c, n)) for s, c, n in con.execute("SELECT status, COUNT(*), SUM(n) FROM ingredient_lines GROUP BY 1")))
all_lines = sum(c for c, _ in rows.values())
all_uses = sum(n for _, n in rows.values())
for status, name in [(1, "done"), (0, "to do"), (-1, "failed"), (-2, "skipped (too long)")]:
    c, n = rows.get(status, (0, 0))
    print(f"{name:20} {c:>10,} distinct lines ({c / all_lines:6.1%})   covering {n:>10,} recipe lines ({n / all_uses:6.1%})")

done                  1,969,444 distinct lines ( 95.4%)   covering  5,944,650 recipe lines ( 98.4%)
to do                    94,432 distinct lines (  4.6%)   covering     94,432 recipe lines (  1.6%)
failed                        1 distinct lines (  0.0%)   covering          1 recipe lines (  0.0%)
skipped (too long)        1,374 distinct lines (  0.1%)   covering      1,391 recipe lines (  0.0%)


In [ ]:
# Step 3: one row per ingredient of every recipe, with cleaned-up units. Rebuilt from scratch every time (~2 min).
# Only lines that are done get rows, so this can run while step 2 is still going.

# unit -> (factor for the amount, new unit). Everything not listed stays as the model returned it.
UNIT_FIX = {
    "dag": (10, "g"), "dkg": (10, "g"), "gr": (1, "g"), "gram": (1, "g"), "grams": (1, "g"), "gramm": (1, "g"),
    "kilogram": (1, "kg"), "cl": (10, "ml"), "dl": (100, "ml"), "liter": (1, "l"), "litre": (1, "l"),
    "spoon": (1, "tbsp"), "tablespoon": (1, "tbsp"), "ss": (1, "tbsp"), "el": (1, "tbsp"), "msk": (1, "tbsp"),
    "teaspoon": (1, "tsp"), "tl": (1, "tsp"), "tsk": (1, "tsp"), "krm": (1, "ml"), "cc": (1, "ml"),
    "pieces": (1, "piece"), "pcs": (1, "piece"), "pc": (1, "piece"), "null": (1, None), "none": (1, None),
}


def clean(name, amount, unit):
    name = " ".join(name.split()).lower() if isinstance(name, str) else None
    if not name or name in ("null", "none") or len(name) > 60:
        return None
    if not isinstance(amount, (int, float)) or isinstance(amount, bool) or amount < 0:
        amount = None
    unit = " ".join(unit.split()).lower() if isinstance(unit, str) and unit.strip() else None
    if unit in UNIT_FIX:
        factor, unit = UNIT_FIX[unit]
        amount = amount * factor if amount is not None else None
    return name, amount, unit


t = time.time()
items = {line: (line_id, json.loads(raw)) for line_id, line, raw in
         con.execute("SELECT id, line, items FROM ingredient_lines WHERE status = 1")}
with con:
    con.executescript("""
    DROP TABLE IF EXISTS recipe_ingredients;
    CREATE TABLE recipe_ingredients (
        recipe_id INTEGER NOT NULL,  -- recipes.rowid
        pos INTEGER NOT NULL,        -- position of the line in the recipe
        line_id INTEGER NOT NULL,    -- ingredient_lines.id
        name TEXT NOT NULL,
        amount REAL,
        unit TEXT
    );
    """)
    rows, complete, recipes = [], 0, 0
    for recipe_id, ingredients in con.execute("SELECT rowid, ingredients FROM recipes"):
        recipes += 1
        lines = [norm(l) for l in json.loads(ingredients)]
        complete += all(l in items or not l or len(l) > MAX_LINE_CHARS for l in lines)
        for pos, line in enumerate(lines):
            if line in items:
                line_id, entries = items[line]
                for entry in entries:
                    if len(entry) == 3 and (c := clean(*entry)):
                        rows.append((recipe_id, pos, line_id, *c))
        if len(rows) > 200_000:
            con.executemany("INSERT INTO recipe_ingredients VALUES (?, ?, ?, ?, ?, ?)", rows)
            rows = []
    con.executemany("INSERT INTO recipe_ingredients VALUES (?, ?, ?, ?, ?, ?)", rows)
    con.execute("CREATE INDEX recipe_ingredients_recipe ON recipe_ingredients(recipe_id)")
    con.execute("CREATE INDEX recipe_ingredients_name ON recipe_ingredients(name)")
del items
print(f"{time.time() - t:.0f} s, {complete:,}/{recipes:,} recipes have every line processed")
print(con.execute("SELECT COUNT(*), COUNT(DISTINCT name) FROM recipe_ingredients").fetchone(), "rows, distinct names")
print("most common:", con.execute(
    "SELECT name, COUNT(*) FROM recipe_ingredients GROUP BY 1 ORDER BY 2 DESC LIMIT 20").fetchall())
print("units:", con.execute(
    "SELECT unit, COUNT(*) FROM recipe_ingredients GROUP BY 1 ORDER BY 2 DESC LIMIT 25").fetchall())

42 s, 572,941/605,440 recipes have every line processed
(5976076, 55163) rows, distinct names
most common: [('salt', 346320), ('sugar', 182024), ('garlic', 160233), ('water', 153656), ('egg', 148870), ('butter', 142248), ('onion', 134197), ('flour', 120261), ('pepper', 113941), ('olive oil', 110296), ('milk', 86333), ('black pepper', 79127), ('oil', 61871), ('soy sauce', 54486), ('carrot', 54369), ('parsley', 52828), ('tomato', 52156), ('lemon juice', 51302), ('vegetable oil', 49815), ('baking powder', 47802)]
units: [('piece', 1138347), (None, 1046007), ('g', 912798), ('tbsp', 767660), ('tsp', 702409), ('cup', 546029), ('ml', 245414), ('package', 98029), ('pinch', 83203), ('clove', 78315), ('can', 54386), ('kg', 48947), ('bunch', 34243), ('l', 29372), ('lb', 23137), ('slice', 21819), ('sprig', 20568), ('handful', 18912), ('sheet', 16982), ('oz', 15116), ('stalk', 11540), ('leaf', 10948), ('cm', 9621), ('inch', 4163), ('drop', 3688)]
